# CRN_Light (Merged) — MVDR Post-Enhancement Training

This notebook is a **single-file merge** of your:
- `crn_light.py` (model)
- `stft_utils.py` (STFT/iSTFT)
- `dataset.py` (MVDRDataset)
- `losses.py` (losses)
- `train_crn.py` (training loop)

Changes requested:
1) **Checkpointing + auto-resume**: saves every `SAVE_EVERY_EPOCHS`, and if you rerun, it loads the latest checkpoint and resumes from the next epoch.
2) **Progress bars**: updates after every batch and **persists** when finished (does not vanish).

On Kaggle, set `MVDR_DIR` / `CLEAN_DIR` to your dataset paths (usually under `/kaggle/input/...`).

In [1]:
# (Optional) Kaggle usually has these already. Uncomment if needed.
# !pip -q install soundfile tqdm

import os
import re
import glob
import math
from dataclasses import dataclass

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset

from tqdm.auto import tqdm

try:
    import torchaudio
    _HAS_TORCHAUDIO = True
except ModuleNotFoundError:
    torchaudio = None
    _HAS_TORCHAUDIO = False

try:
    import soundfile as sf
    _HAS_SOUNDFILE = True
except ModuleNotFoundError:
    sf = None
    _HAS_SOUNDFILE = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', DEVICE)
print('Torch:', torch.__version__)

Device: cpu
Torch: 2.10.0


/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/custom_model_1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model (from `crn_light.py`)

In [2]:
class DepthwiseSeparableConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size=(3, 3),
        stride=(1, 1),
        padding=(1, 1),
        bias: bool = True,
    ):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
            bias=bias,
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=bias)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x


class InvertedResidual2d(nn.Module):
    def __init__(
        self,
        channels: int,
        expansion: int = 4,
        kernel_size=(3, 3),
        causal: bool = False,
        bias: bool = True,
    ):
        super().__init__()
        hidden = channels * expansion
        self.causal = causal
        self.kernel_size = kernel_size

        self.expand = nn.Conv2d(channels, hidden, kernel_size=1, bias=bias)
        self.dw = nn.Conv2d(
            hidden,
            hidden,
            kernel_size=kernel_size,
            stride=1,
            padding=0 if causal else (kernel_size[0] // 2, kernel_size[1] // 2),
            groups=hidden,
            bias=bias,
        )
        self.project = nn.Conv2d(hidden, channels, kernel_size=1, bias=bias)

    def forward(self, x):
        residual = x
        x = F.relu(self.expand(x))

        if self.causal:
            # Causal along time (last dim). Keep freq padding symmetric.
            pad_f = self.kernel_size[0] // 2
            pad_t = self.kernel_size[1] - 1
            x = F.pad(x, (pad_t, 0, pad_f, pad_f))

        x = F.relu(self.dw(x))
        x = self.project(x)
        return x + residual


class CRN_Light(nn.Module):
    """
    Tiny CRN-like U-Net for MVDR post-enhancement
    ~0.15M parameters (target for on-device)
    Input:  [B, 2, F, T] where channels = [real, imag]
    Output: [B, 2, F, T] complex mask [m_r, m_i]
    """

    def __init__(
        self,
        freq_bins: int = 257,
        in_channels: int = 2,
        out_channels: int = 2,
        base_channels: int = 16,
        bottleneck_channels: int = 48,
        bottleneck_blocks: int = 6,
        bottleneck_expansion: int = 4,
        causal: bool = False,
        mask_mag_max: float = 2.0,
        mask_imag_max: float = 1.0,
    ):
        super().__init__()

        c1 = base_channels
        c2 = int(round(base_channels * 1.5))
        c3 = int(round(base_channels * 2.0))
        c4 = bottleneck_channels

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.mask_mag_max = float(mask_mag_max)
        self.mask_imag_max = float(mask_imag_max)

        self.enc1 = DepthwiseSeparableConv2d(in_channels, c1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        self.enc2 = DepthwiseSeparableConv2d(c1, c2, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))
        self.enc3 = DepthwiseSeparableConv2d(c2, c3, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))
        self.enc4 = DepthwiseSeparableConv2d(c3, c4, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))

        # Three strided layers reduce F as ceil(F/8).
        self.freq_reduced = (freq_bins + 7) // 8

        self.bottleneck = nn.Sequential(
            *[
                InvertedResidual2d(
                    channels=c4,
                    expansion=bottleneck_expansion,
                    kernel_size=(3, 3),
                    causal=causal,
                )
                for _ in range(bottleneck_blocks)
            ]
)

        self.up4 = nn.Upsample(scale_factor=(2, 1), mode="nearest")
        self.dec4 = DepthwiseSeparableConv2d(c4 + c3, c3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

        self.up3 = nn.Upsample(scale_factor=(2, 1), mode="nearest")
        self.dec3 = DepthwiseSeparableConv2d(c3 + c2, c2, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

        self.up2 = nn.Upsample(scale_factor=(2, 1), mode="nearest")
        self.dec2 = DepthwiseSeparableConv2d(c2 + c1, c1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

        self.out = nn.Conv2d(c1, out_channels, kernel_size=1)

        # Identity-friendly init for the mask mapping below:
        # - raw_r = 0 -> sigmoid(0)=0.5 -> m_r = mask_mag_max * 0.5 (==1.0 when mask_mag_max=2)
        # - raw_i = 0 -> tanh(0)=0 -> m_i = 0
        nn.init.zeros_(self.out.weight)
        if self.out.bias is not None:
            nn.init.zeros_(self.out.bias)

    def forward(self, x):
        # x: [B,2,F,T]

        def _align_to(ref, y):
            """Crop/pad y on (freq,time) dims to match ref."""
            ref_f, ref_t = ref.shape[2], ref.shape[3]
            y_f, y_t = y.shape[2], y.shape[3]

            if y_f > ref_f:
                y = y[:, :, :ref_f, :]
            elif y_f < ref_f:
                y = F.pad(y, (0, 0, 0, ref_f - y_f))

            if y_t > ref_t:
                y = y[:, :, :, :ref_t]
            elif y_t < ref_t:
                y = F.pad(y, (0, ref_t - y_t, 0, 0))

            return y

        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(e1))
        e3 = F.relu(self.enc3(e2))
        e4 = F.relu(self.enc4(e3))

        b = self.bottleneck(e4)

        d4 = self.up4(b)
        d4 = _align_to(e3, d4)
        d4 = F.relu(self.dec4(torch.cat([d4, e3], dim=1)))

        d3 = self.up3(d4)
        d3 = _align_to(e2, d3)
        d3 = F.relu(self.dec3(torch.cat([d3, e2], dim=1)))

        d2 = self.up2(d3)
        d2 = _align_to(e1, d2)
        d2 = F.relu(self.dec2(torch.cat([d2, e1], dim=1)))

        # IMPORTANT FIX: allow strong suppression (mask magnitude down to near 0).
        # raw has 2 channels: [raw_r, raw_i].
        raw = self.out(d2)
        if raw.shape[1] == 2:
            raw_r = raw[:, 0:1]
            raw_i = raw[:, 1:2]

            # Real-part (magnitude-like) mask in [0, mask_mag_max].
            # With mask_mag_max=2, init gives m_r=1.0, but it can go close to 0 for heavy attenuation.
            m_r = self.mask_mag_max * torch.sigmoid(raw_r)

            # Imag mask in [-mask_imag_max, mask_imag_max].
            m_i = self.mask_imag_max * torch.tanh(raw_i)
            return torch.cat([m_r, m_i], dim=1)

        # Fallback for non-complex masks
        return torch.tanh(raw)

## STFT utils (from `stft_utils.py`)

In [3]:
def stft_mag(wav, device=None):
    if device is None:
        device = wav.device
    window = torch.sqrt(torch.hann_window(256, periodic=True)).to(device)

    spec = torch.stft(
        wav,
        n_fft=512,
        hop_length=128,
        win_length=256,
        window=window,
        center=True,
        return_complex=True,
    )
    return torch.abs(spec), spec


def istft(spec, device=None, length=None):
    if device is None:
        device = spec.device
    window = torch.sqrt(torch.hann_window(256, periodic=True)).to(device)

    return torch.istft(
        spec,
        n_fft=512,
        hop_length=128,
        win_length=256,
        window=window,
        center=True,
        length=length,
    )

## Dataset (from `dataset.py`)

In [10]:
def _load_wav(path: str, target_sr: int) -> torch.Tensor:
    # Prefer soundfile when available: lightweight + avoids optional torchaudio decoder deps.
    if _HAS_SOUNDFILE:
        data, sr = sf.read(path, dtype="float32", always_2d=True)  # [N, C]
        if sr != target_sr:
            if _HAS_TORCHAUDIO:
                wav = torch.from_numpy(np.asarray(data).T)  # [C, N]
                wav = torchaudio.functional.resample(wav, sr, target_sr)
                sr = target_sr
            else:
                raise ValueError(
                    f"Sample rate mismatch for {path}: got {sr}, expected {target_sr}. "
                    "Install torchaudio to enable resampling."
                )
        else:
            wav = torch.from_numpy(np.asarray(data).T)
    elif _HAS_TORCHAUDIO:
        wav, sr = torchaudio.load(path)
        if sr != target_sr:
            wav = torchaudio.functional.resample(wav, sr, target_sr)
        wav = wav.to(torch.float32)
    else:
        raise ModuleNotFoundError("Missing audio backend. Install soundfile or torchaudio.")

    if wav.dim() != 2:
        raise RuntimeError(f"Expected waveform shape [C,N], got {tuple(wav.shape)} for {path}")

    # Mix to mono if multi-channel
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0)
    else:
        wav = wav.squeeze(0)

    return wav


class MVDRDataset(Dataset):
    def __init__(
        self,
        mvdr_dir,
        clean_dir,
        sample_rate=16000,
        strict_pairing_check: bool = True,
        verbose_pairing_warnings: bool = True,
    ):
        self.mvdr_dir = mvdr_dir
        self.clean_dir = clean_dir
        self.sr = sample_rate
        self.strict_pairing_check = bool(strict_pairing_check)
        self.verbose_pairing_warnings = bool(verbose_pairing_warnings)

        self.mvdr_files = sorted([f for f in os.listdir(mvdr_dir) if f.lower().endswith(".wav")])

        # Index clean references by stem (supports .wav/.flac)
        self.clean_index = {}
        for f in os.listdir(clean_dir):
            lower = f.lower()
            if not (lower.endswith(".wav") or lower.endswith(".flac")):
                continue
            stem, _ = os.path.splitext(f)
            self.clean_index[stem] = os.path.join(clean_dir, f)

        if self.strict_pairing_check:
            # Fail fast if any MVDR file cannot be paired uniquely.
            for mvdr_name in self.mvdr_files:
                clean_path = self._resolve_clean_path(mvdr_name)
                if clean_path is None:
                    raise FileNotFoundError(
                        f"No clean match found for MVDR '{mvdr_name}'. "
                        f"Check naming / pairing logic in _get_clean_stem()."
                    )

    def _get_clean_stem(self, mvdr_name: str) -> str:
        # Robust stem extraction. Prefer patterns like '1_part13' at start, else fallback to first 2 '_' chunks.
        base = os.path.splitext(mvdr_name)[0]
        m = re.match(r"^(\d+_part\d+)", base)
        if m:
            return m.group(1)
        # Fallback: 1_part13_A_female_only.wav → 1_part13
        return "_".join(base.split("_")[:2])

    def _resolve_clean_path(self, mvdr_name: str):
        clean_stem = self._get_clean_stem(mvdr_name)
        # Exact match first
        if clean_stem in self.clean_index:
            return self.clean_index[clean_stem]
        # Prefix match fallback (handles extra suffixes in clean names)
        candidates = [stem for stem in self.clean_index.keys() if stem.startswith(clean_stem)]
        if len(candidates) == 1:
            return self.clean_index[candidates[0]]
        if len(candidates) > 1 and self.verbose_pairing_warnings:
            print(f"Warning: multiple clean candidates for {mvdr_name} (stem {clean_stem}): {candidates[:5]}...")
        return None

    def __len__(self):
        return len(self.mvdr_files)

    def __getitem__(self, idx):
        mvdr_name = self.mvdr_files[idx]
        mvdr_path = os.path.join(self.mvdr_dir, mvdr_name)
        clean_path = self._resolve_clean_path(mvdr_name)

        if not os.path.isfile(mvdr_path):
            raise FileNotFoundError(f"Missing MVDR wav: {mvdr_path}")
        if clean_path is None or not os.path.isfile(clean_path):
            raise FileNotFoundError(
                f"Missing clean reference for '{mvdr_name}'. "
                f"Check naming: expected stem like '{self._get_clean_stem(mvdr_name)}.wav/.flac' in {self.clean_dir}"
            )

        mvdr_wav = _load_wav(mvdr_path, self.sr)
        clean_wav = _load_wav(clean_path, self.sr)

        # Ensure paired signals have identical length
        min_len = min(mvdr_wav.shape[-1], clean_wav.shape[-1])
        mvdr_wav = mvdr_wav[:min_len]
        clean_wav = clean_wav[:min_len]

        # Normalize both using MVDR scale (preserves relative loudness).
        scale = mvdr_wav.std() + 1e-8
        mvdr_wav = mvdr_wav / scale
        clean_wav = clean_wav / scale

        return mvdr_wav, clean_wav

## Losses (from `losses.py`)

In [11]:
def log_mag_loss(est, ref):
    return F.l1_loss(torch.log(est + 1e-8), torch.log(ref + 1e-8))


def si_sdr_loss(est, ref, eps=1e-8):
    ref = ref - ref.mean(dim=-1, keepdim=True)
    est = est - est.mean(dim=-1, keepdim=True)

    proj = (torch.sum(est * ref, dim=-1, keepdim=True) * ref) / (torch.sum(ref ** 2, dim=-1, keepdim=True) + eps)
    noise = est - proj
    ratio = torch.sum(proj ** 2, dim=-1) / (torch.sum(noise ** 2, dim=-1) + eps)

    return (-10.0 * torch.log10(ratio + eps)).mean()


def _stft(x, n_fft, hop_length, win_length, window):
    return torch.stft(
        x,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=True,
        return_complex=True,
    )


def mrstft_loss(
    est_wav: torch.Tensor,
    ref_wav: torch.Tensor,
    fft_sizes=(512, 1024, 2048),
    hop_ratios=(0.25, 0.25, 0.25),
    win_lengths=None,
    eps: float = 1e-8,
):
    """Multi-resolution STFT loss: spectral convergence + log-mag L1.

    Shapes: est_wav/ref_wav: [B, T]
    """
    if win_lengths is None:
        win_lengths = fft_sizes
    device = est_wav.device

    sc_total = 0.0
    mag_total = 0.0
    n = 0

    for n_fft, hop_ratio, win_length in zip(fft_sizes, hop_ratios, win_lengths):
        hop = int(round(n_fft * hop_ratio))
        window = torch.hann_window(win_length, periodic=True, device=device)

        est = _stft(est_wav, n_fft=n_fft, hop_length=hop, win_length=win_length, window=window)
        ref = _stft(ref_wav, n_fft=n_fft, hop_length=hop, win_length=win_length, window=window)

        est_mag = torch.abs(est)
        ref_mag = torch.abs(ref)

        sc = torch.norm(ref_mag - est_mag, p="fro") / (torch.norm(ref_mag, p="fro") + eps)
        mag = F.l1_loss(torch.log(est_mag + eps), torch.log(ref_mag + eps))

        sc_total = sc_total + sc
        mag_total = mag_total + mag
        n += 1

    sc_total = sc_total / max(1, n)
    mag_total = mag_total / max(1, n)
    return sc_total + mag_total

## Training + Checkpointing (merged from `train_crn.py`)

Key additions:
- Saves a full checkpoint dict (model + optimizer + epoch) every `SAVE_EVERY_EPOCHS`.
- On rerun, auto-loads the **latest epoch checkpoint** and resumes.
- Uses `tqdm.auto` with `leave=True`, `miniters=1`, `mininterval=0` so batch progress updates and persists.

In [12]:
# -------------------- CONFIG (edit values here) --------------------
# For Kaggle, set these to your dataset folder paths.
# MVDR_DIR = '/kaggle/input/post-mvdr-snr-5db/mvdr_outputs'
# CLEAN_DIR = '/kaggle/input/post-mvdr-snr-5db/true_labels'
MVDR_DIR = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/custom_model_1/mvdr_outputs'
CLEAN_DIR = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/custom_model_1/true_labels'

SAMPLE_RATE = 16000

BATCH_SIZE = 8
EPOCHS = 60
LR = 2e-4

SISDR_WARMUP_EPOCHS = 0
SI_SDR_WEIGHT_MAX = 1.0
MRSTFT_WEIGHT = 0.3
LOGMAG_WEIGHT = 0.2
IMPROVEMENT_HINGE_WEIGHT = 0.5

# Additional loss to reduce residual noise during silence/low-energy regions.
SILENCE_LOSS_WEIGHT = 0.15
SILENCE_RMS_THRESHOLD = 0.01  # threshold in normalized waveform space
SILENCE_FRAME_LEN = 512
SILENCE_HOP = 128

# Train/Val/Test split ratios (must sum to 1.0)
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

SEED = 1337

NUM_WORKERS = 4 if DEVICE == 'cuda' else 0

SEGMENT_SECONDS = 2.0
SEGMENT_SAMPLES = int(SAMPLE_RATE * SEGMENT_SECONDS)
SEGMENT_CANDIDATES = 8

# FIX (energy-biased cropping): mix high-energy + random + low-energy crops.
CROP_STRATEGY_PROBS = {
    'high_energy': 0.6,
    'random': 0.25,
    'low_energy': 0.15,
}

VAL_FULL_UTTERANCE = True

MAX_TRAIN_BATCHES_PER_EPOCH = None
MAX_VAL_BATCHES_PER_EPOCH = None

SAVE_EVERY_EPOCHS = 1
CHECKPOINT_PREFIX = 'crn'
CHECKPOINT_DIR = 'checkpoints'
CHECKPOINT_LOAD_DIR = CHECKPOINT_DIR  ### edit here
AUTO_RESUME = True

# Test set saving - saves actual audio files + JSON manifest
TEST_SET_DIR = os.path.join(CHECKPOINT_DIR, 'test_set')
TEST_SET_MVDR_DIR = os.path.join(TEST_SET_DIR, 'mvdr')
TEST_SET_CLEAN_DIR = os.path.join(TEST_SET_DIR, 'clean')
TEST_SET_MANIFEST_PATH = os.path.join(TEST_SET_DIR, 'manifest.json')

# Early stopping: stop if validation improvement does not improve for N epochs.
EARLY_STOPPING = True
EARLY_STOP_PATIENCE = 6
EARLY_STOP_MIN_DELTA_DB = 0.0  # require at least this many dB improvement over best

# Dataset pairing sanity checks
STRICT_PAIRING_CHECK = True
PAIRING_SANITY_PRINT = 5  # number of random pairs to print at startup
# -------------------------------------------------------------------

import json
import shutil

def _seed_everything(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _normalize_probs(d: dict):
    s = float(sum(d.values()))
    if s <= 0:
        raise ValueError('CROP_STRATEGY_PROBS must sum to > 0')
    return {k: float(v) / s for k, v in d.items()}


def _choose_crop_strategy() -> str:
    probs = _normalize_probs(CROP_STRATEGY_PROBS)
    keys = list(probs.keys())
    p = np.array([probs[k] for k in keys], dtype=np.float64)
    return str(np.random.choice(keys, p=p))


def _candidate_starts(max_start: int, k: int):
    if max_start <= 0:
        return [0]
    k = int(min(max(1, k), max_start + 1))
    return torch.randint(low=0, high=max_start + 1, size=(k,)).tolist()


def _pick_start_by_energy(clean_wav: torch.Tensor, max_start: int, mode: str) -> int:
    # mode: 'high_energy' picks max-energy; 'low_energy' picks min-energy
    if max_start <= 0:
        return 0

    starts = _candidate_starts(max_start, SEGMENT_CANDIDATES)
    best_start = starts[0]
    best_val = None

    for s in starts:
        seg = clean_wav[s : s + SEGMENT_SAMPLES]
        energy = torch.mean(seg * seg).item()
        if best_val is None:
            best_val = energy
            best_start = s
            continue
        if mode == 'high_energy' and energy > best_val:
            best_val = energy
            best_start = s
        if mode == 'low_energy' and energy < best_val:
            best_val = energy
            best_start = s

    return int(best_start)


def _pad_collate(batch):
    mvdr_list, clean_list = zip(*batch)
    mvdr_out = []
    clean_out = []

    for mvdr_wav, clean_wav in zip(mvdr_list, clean_list):
        if SEGMENT_SAMPLES is None:
            mvdr_out.append(mvdr_wav)
            clean_out.append(clean_wav)
            continue

        length = mvdr_wav.shape[-1]
        if length >= SEGMENT_SAMPLES:
            max_start = length - SEGMENT_SAMPLES
            start = max_start // 2
            mvdr_wav = mvdr_wav[start : start + SEGMENT_SAMPLES]
            clean_wav = clean_wav[start : start + SEGMENT_SAMPLES]
        else:
            pad = SEGMENT_SAMPLES - length
            mvdr_wav = torch.nn.functional.pad(mvdr_wav, (0, pad))
            clean_wav = torch.nn.functional.pad(clean_wav, (0, pad))

        mvdr_out.append(mvdr_wav)
        clean_out.append(clean_wav)

    return torch.stack(mvdr_out, dim=0), torch.stack(clean_out, dim=0)


def _train_collate(batch):
    # FIX: not always high-energy; include random and low-energy segments too.
    mvdr_list, clean_list = zip(*batch)
    mvdr_out = []
    clean_out = []

    for mvdr_wav, clean_wav in zip(mvdr_list, clean_list):
        if SEGMENT_SAMPLES is None:
            mvdr_out.append(mvdr_wav)
            clean_out.append(clean_wav)
            continue

        length = mvdr_wav.shape[-1]
        if length >= SEGMENT_SAMPLES:
            max_start = length - SEGMENT_SAMPLES
            strategy = _choose_crop_strategy()

            if strategy == 'random':
                start = int(torch.randint(low=0, high=max_start + 1, size=(1,)).item())
            elif strategy == 'low_energy':
                start = _pick_start_by_energy(clean_wav, max_start, mode='low_energy')
            else:
                start = _pick_start_by_energy(clean_wav, max_start, mode='high_energy')

            mvdr_wav = mvdr_wav[start : start + SEGMENT_SAMPLES]
            clean_wav = clean_wav[start : start + SEGMENT_SAMPLES]
        else:
            pad = SEGMENT_SAMPLES - length
            mvdr_wav = torch.nn.functional.pad(mvdr_wav, (0, pad))
            clean_wav = torch.nn.functional.pad(clean_wav, (0, pad))

        mvdr_out.append(mvdr_wav)
        clean_out.append(clean_wav)

    return torch.stack(mvdr_out, dim=0), torch.stack(clean_out, dim=0)


def _validate_paths():
    if not os.path.isdir(MVDR_DIR):
        raise FileNotFoundError(f'MVDR_DIR not found: {MVDR_DIR}')
    if not os.path.isdir(CLEAN_DIR):
        raise FileNotFoundError(f'CLEAN_DIR not found: {CLEAN_DIR}')

    mvdr_wavs = glob.glob(os.path.join(MVDR_DIR, '*.wav'))
    clean_wavs = glob.glob(os.path.join(CLEAN_DIR, '*.wav')) + glob.glob(os.path.join(CLEAN_DIR, '*.flac'))
    if len(mvdr_wavs) == 0:
        raise RuntimeError(f'No .wav files found in {MVDR_DIR}')
    if len(clean_wavs) == 0:
        raise RuntimeError(f'No .wav/.flac files found in {CLEAN_DIR}')


def _validate_split_ratios():
    total = TRAIN_RATIO + VAL_RATIO + TEST_RATIO
    if not (0.99 <= total <= 1.01):
        raise ValueError(f'TRAIN_RATIO + VAL_RATIO + TEST_RATIO must sum to 1.0, got {total:.3f}')


def save_test_set_files(dataset, test_indices: list):
    """Save test set audio files (MVDR + clean) and manifest for later use."""
    os.makedirs(TEST_SET_MVDR_DIR, exist_ok=True)
    os.makedirs(TEST_SET_CLEAN_DIR, exist_ok=True)
    
    test_files = []
    print(f'Saving {len(test_indices)} test set files...')
    
    for idx in tqdm(test_indices, desc='Copying test files'):
        mvdr_name = dataset.mvdr_files[idx]
        mvdr_src = os.path.join(dataset.mvdr_dir, mvdr_name)
        clean_src = dataset._resolve_clean_path(mvdr_name)
        
        if clean_src is None:
            print(f'  Warning: skipping {mvdr_name} - no clean match')
            continue
        
        clean_name = os.path.basename(clean_src)
        
        # Copy MVDR file
        mvdr_dst = os.path.join(TEST_SET_MVDR_DIR, mvdr_name)
        shutil.copy2(mvdr_src, mvdr_dst)
        
        # Copy clean file
        clean_dst = os.path.join(TEST_SET_CLEAN_DIR, clean_name)
        if not os.path.exists(clean_dst):  # Avoid duplicate copies
            shutil.copy2(clean_src, clean_dst)
        
        test_files.append({
            'index': int(idx),
            'mvdr_file': mvdr_name,
            'clean_file': clean_name,
        })
    
    # Save manifest
    manifest = {
        'seed': SEED,
        'train_ratio': TRAIN_RATIO,
        'val_ratio': VAL_RATIO,
        'test_ratio': TEST_RATIO,
        'total_samples': len(dataset),
        'test_count': len(test_files),
        'test_indices': [int(i) for i in test_indices],
        'test_files': test_files,
        'original_mvdr_dir': MVDR_DIR,
        'original_clean_dir': CLEAN_DIR,
        'saved_mvdr_dir': TEST_SET_MVDR_DIR,
        'saved_clean_dir': TEST_SET_CLEAN_DIR,
    }
    
    with open(TEST_SET_MANIFEST_PATH, 'w') as f:
        json.dump(manifest, f, indent=2)
    
    print(f'Saved test set:')
    print(f'  MVDR files:  {TEST_SET_MVDR_DIR}')
    print(f'  Clean files: {TEST_SET_CLEAN_DIR}')
    print(f'  Manifest:    {TEST_SET_MANIFEST_PATH}')
    return manifest


def load_test_set_info(load_path: str = TEST_SET_MANIFEST_PATH) -> dict:
    """Load previously saved test set manifest."""
    with open(load_path, 'r') as f:
        return json.load(f)


def _make_loaders():
    _validate_split_ratios()
    
    dataset = MVDRDataset(
        MVDR_DIR,
        CLEAN_DIR,
        sample_rate=SAMPLE_RATE,
        strict_pairing_check=STRICT_PAIRING_CHECK,
        verbose_pairing_warnings=True,
    )
    if len(dataset) < 3:
        raise RuntimeError(f'Need at least 3 samples to split train/val/test, got {len(dataset)}')

    if PAIRING_SANITY_PRINT:
        print('Pairing sanity check (first few samples):')
        for i in range(min(PAIRING_SANITY_PRINT, len(dataset))):
            mvdr_name = dataset.mvdr_files[i]
            clean_path = dataset._resolve_clean_path(mvdr_name)
            print(f'  {mvdr_name} -> {os.path.basename(clean_path) if clean_path else None}')

    # Compute split sizes
    n_total = len(dataset)
    n_train = int(round(n_total * TRAIN_RATIO))
    n_val = int(round(n_total * VAL_RATIO))
    n_test = n_total - n_train - n_val  # Remainder goes to test
    
    # Ensure minimum 1 sample per split
    n_train = max(1, n_train)
    n_val = max(1, n_val)
    n_test = max(1, n_test)
    
    # Adjust if we exceed total
    while n_train + n_val + n_test > n_total:
        if n_train > 1:
            n_train -= 1
        elif n_val > 1:
            n_val -= 1
        else:
            n_test -= 1

    # Random split with no duplicates using random_split
    generator = torch.Generator().manual_seed(SEED)
    train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test], generator=generator)
    
    # Save test set files (actual audio files + manifest)
    test_indices = list(test_set.indices)
    save_test_set_files(dataset, test_indices)

    print(f'\nDataset split (seed={SEED}, no duplicates):')
    print(f'  Train: {len(train_set)} samples ({100*len(train_set)/n_total:.1f}%)')
    print(f'  Val:   {len(val_set)} samples ({100*len(val_set)/n_total:.1f}%)')
    print(f'  Test:  {len(test_set)} samples ({100*len(test_set)/n_total:.1f}%)')

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
        collate_fn=_train_collate,
    )

    if VAL_FULL_UTTERANCE:
        val_loader = DataLoader(
            val_set,
            batch_size=1,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE == 'cuda'),
        )
        test_loader = DataLoader(
            test_set,
            batch_size=1,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE == 'cuda'),
        )
    else:
        val_loader = DataLoader(
            val_set,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE == 'cuda'),
            collate_fn=_pad_collate,
        )
        test_loader = DataLoader(
            test_set,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE == 'cuda'),
            collate_fn=_pad_collate,
        )

    return train_loader, val_loader, test_loader


def _sisdr_weight_for_epoch(epoch_idx: int) -> float:
    if SISDR_WARMUP_EPOCHS <= 0:
        return SI_SDR_WEIGHT_MAX
    if epoch_idx <= SISDR_WARMUP_EPOCHS:
        return 0.0
    ramp = min(1.0, (epoch_idx - SISDR_WARMUP_EPOCHS) / 5.0)
    return SI_SDR_WEIGHT_MAX * ramp


def _frame_rms(wav_bt: torch.Tensor, frame_len: int, hop: int, eps: float = 1e-8) -> torch.Tensor:
    """Compute frame RMS over waveform. wav_bt: [B,T] -> [B,frames]."""
    if wav_bt.dim() != 2:
        raise ValueError(f'Expected [B,T], got {tuple(wav_bt.shape)}')

    b, t = wav_bt.shape
    if t < frame_len:
        pad = frame_len - t
        wav_bt = torch.nn.functional.pad(wav_bt, (0, pad))

    x = wav_bt.unsqueeze(1)  # [B,1,T]
    kernel = torch.ones(1, 1, frame_len, device=wav_bt.device) / float(frame_len)
    mean_sq = torch.nn.functional.conv1d(x * x, kernel, stride=hop)  # [B,1,frames]
    return torch.sqrt(mean_sq.squeeze(1) + eps)


def _silence_loss(est_wav_bt: torch.Tensor, ref_wav_bt: torch.Tensor) -> torch.Tensor:
    """Penalize residual energy where reference is silent/very low energy."""
    ref_rms = _frame_rms(ref_wav_bt, frame_len=SILENCE_FRAME_LEN, hop=SILENCE_HOP)
    est_rms = _frame_rms(est_wav_bt, frame_len=SILENCE_FRAME_LEN, hop=SILENCE_HOP)

    silent = (ref_rms < SILENCE_RMS_THRESHOLD).to(est_rms.dtype)
    denom = silent.sum().clamp(min=1.0)
    return (est_rms * silent).sum() / denom


def _forward_losses(model, mvdr_wav, clean_wav, epoch_idx: int):
    _, mvdr_complex = stft_mag(mvdr_wav, DEVICE)
    clean_mag, _ = stft_mag(clean_wav, DEVICE)

    x = torch.stack([mvdr_complex.real, mvdr_complex.imag], dim=1)  # [B,2,F,T]

    m = model(x)
    m_r = m[:, 0, :, :]
    m_i = m[:, 1, :, :]

    x_r = mvdr_complex.real
    x_i = mvdr_complex.imag

    y_r = m_r * x_r - m_i * x_i
    y_i = m_r * x_i + m_i * x_r
    est_complex = torch.complex(y_r, y_i)

    est_wav = istft(est_complex, DEVICE, length=clean_wav.shape[-1])

    est_mag = torch.abs(est_complex).unsqueeze(1)
    clean_mag = clean_mag.unsqueeze(1)

    loss_logmag = log_mag_loss(est_mag, clean_mag)
    loss_sisdr = si_sdr_loss(est_wav, clean_wav)
    loss_mrstft = mrstft_loss(est_wav, clean_wav)

    # FIX: additional penalty for residual noise during silence
    loss_silence = _silence_loss(est_wav, clean_wav)

    baseline_sisdr_loss = si_sdr_loss(mvdr_wav, clean_wav)
    improvement_hinge = torch.relu(loss_sisdr - baseline_sisdr_loss)

    si_sdr_weight = _sisdr_weight_for_epoch(epoch_idx)

    total_loss = (
        si_sdr_weight * loss_sisdr
        + MRSTFT_WEIGHT * loss_mrstft
        + LOGMAG_WEIGHT * loss_logmag
        + SILENCE_LOSS_WEIGHT * loss_silence
        + IMPROVEMENT_HINGE_WEIGHT * improvement_hinge
    )

    return total_loss, loss_logmag, loss_sisdr, baseline_sisdr_loss, loss_silence


def _make_pbar(iterable, desc: str):
    # Updates after every batch and persists after finishing.
    return tqdm(
        iterable,
        desc=desc,
        leave=True,
        miniters=1,
        mininterval=0.0,
        smoothing=0.0,
        dynamic_ncols=True,
    )


def train_one_epoch(model, optimizer, train_loader, epoch_idx):
    model.train()

    total = 0.0
    total_mag = 0.0
    total_sisdr = 0.0
    total_base_sisdr = 0.0
    total_sil = 0.0
    n = 0

    pbar = _make_pbar(train_loader, desc=f'Train {epoch_idx:03d}')
    for batch_idx, (mvdr_wav, clean_wav) in enumerate(pbar, start=1):
        mvdr_wav = mvdr_wav.to(DEVICE)
        clean_wav = clean_wav.to(DEVICE)

        loss, loss_mag, loss_sisdr, base_sisdr, loss_sil = _forward_losses(model, mvdr_wav, clean_wav, epoch_idx)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total += float(loss.item())
        total_mag += float(loss_mag.item())
        total_sisdr += float(loss_sisdr.item())
        total_base_sisdr += float(base_sisdr.item())
        total_sil += float(loss_sil.item())
        n += 1

        est_db = -(total_sisdr / n)
        base_db = -(total_base_sisdr / n)
        imp_db = est_db - base_db

        pbar.set_postfix(
            loss=f'{total/n:.3f}',
            mag=f'{total_mag/n:.3f}',
            sil=f'{total_sil/n:.4f}',
            sisdr_db=f'{est_db:.2f}',
            imp_db=f'{imp_db:.2f}',
        )

        if MAX_TRAIN_BATCHES_PER_EPOCH is not None and batch_idx >= MAX_TRAIN_BATCHES_PER_EPOCH:
            break

    denom = max(1, n)
    return total / denom, total_mag / denom, total_sisdr / denom, total_base_sisdr / denom, total_sil / denom


@torch.no_grad()
def validate(model, val_loader, epoch_idx):
    model.eval()

    total = 0.0
    total_mag = 0.0
    total_sisdr = 0.0
    total_base_sisdr = 0.0
    total_sil = 0.0
    n = 0

    pbar = _make_pbar(val_loader, desc=f'Val   {epoch_idx:03d}')
    for batch_idx, (mvdr_wav, clean_wav) in enumerate(pbar, start=1):
        mvdr_wav = mvdr_wav.to(DEVICE)
        clean_wav = clean_wav.to(DEVICE)

        loss, loss_mag, loss_sisdr, base_sisdr, loss_sil = _forward_losses(model, mvdr_wav, clean_wav, epoch_idx)

        total += float(loss.item())
        total_mag += float(loss_mag.item())
        total_sisdr += float(loss_sisdr.item())
        total_base_sisdr += float(base_sisdr.item())
        total_sil += float(loss_sil.item())
        n += 1

        est_db = -(total_sisdr / n)
        base_db = -(total_base_sisdr / n)
        imp_db = est_db - base_db

        pbar.set_postfix(
            loss=f'{total/n:.3f}',
            mag=f'{total_mag/n:.3f}',
            sil=f'{total_sil/n:.4f}',
            sisdr_db=f'{est_db:.2f}',
            imp_db=f'{imp_db:.2f}',
        )

        if MAX_VAL_BATCHES_PER_EPOCH is not None and batch_idx >= MAX_VAL_BATCHES_PER_EPOCH:
            break

    denom = max(1, n)
    return total / denom, total_mag / denom, total_sisdr / denom, total_base_sisdr / denom, total_sil / denom


def _checkpoint_paths(epoch: int):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    epoch_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_epoch_{epoch:03d}.pt')
    latest_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_latest.pt')
    return epoch_path, latest_path


def save_checkpoint(epoch: int, model: nn.Module, optimizer: optim.Optimizer):
    epoch_path, latest_path = _checkpoint_paths(epoch)

    payload = {
        'epoch': int(epoch),
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'config': {
            'sample_rate': SAMPLE_RATE,
            'batch_size': BATCH_SIZE,
            'lr': LR,
        },
    }

    torch.save(payload, epoch_path)
    torch.save(payload, latest_path)
    return epoch_path


def _parse_epoch_from_filename(path: str):
    m = re.search(r'_epoch_(\d+)\.pt$', os.path.basename(path))
    if not m:
        return None
    return int(m.group(1))


def find_latest_checkpoint():
    # Prefer explicit latest if present, else scan epoch files.
    latest_path = os.path.join(CHECKPOINT_LOAD_DIR, f'{CHECKPOINT_PREFIX}_latest.pt')
    if os.path.isfile(latest_path):
        return latest_path

    pattern = os.path.join(CHECKPOINT_LOAD_DIR, f'{CHECKPOINT_PREFIX}_epoch_*.pt')
    candidates = glob.glob(pattern)
    if not candidates:
        return None

    best = None
    best_epoch = -1
    for p in candidates:
        ep = _parse_epoch_from_filename(p)
        if ep is None:
            continue
        if ep > best_epoch:
            best_epoch = ep
            best = p

    return best


def maybe_resume(model: nn.Module, optimizer: optim.Optimizer):
    if not AUTO_RESUME:
        return 1

    ckpt_path = find_latest_checkpoint()
    if ckpt_path is None:
        return 1

    ckpt = torch.load(ckpt_path, map_location=DEVICE)

    # Backward compatible: if someone saved raw state_dict only.
    if isinstance(ckpt, dict) and 'model_state' in ckpt:
        model.load_state_dict(ckpt['model_state'])
        if 'optimizer_state' in ckpt:
            try:
                optimizer.load_state_dict(ckpt['optimizer_state'])
            except Exception as e:
                print('Warning: could not load optimizer state:', e)
        last_epoch = int(ckpt.get('epoch', 0))
    else:
        model.load_state_dict(ckpt)
        last_epoch = _parse_epoch_from_filename(ckpt_path) or 0

    print(f'Resuming from checkpoint: {ckpt_path} (epoch {last_epoch})')
    return last_epoch + 1


def main():
    _seed_everything(SEED)
    _validate_paths()

    train_loader, val_loader, test_loader = _make_loaders()

    # Mask range fix is inside CRN_Light now.
    model = CRN_Light().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    start_epoch = maybe_resume(model, optimizer)

    print(f'Device: {DEVICE}')
    print(f'Train/Val/Test sizes: {len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}')
    print(f'Val full utterance: {VAL_FULL_UTTERANCE}')
    print(f'Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')
    print(f'Start epoch: {start_epoch} / Target epochs: {EPOCHS}')

    if start_epoch > EPOCHS:
        print('Nothing to do: latest checkpoint is already >= EPOCHS.')
        return

    best_val_imp_db = -float('inf')
    epochs_no_improve = 0

    for epoch in range(start_epoch, EPOCHS + 1):
        tr_loss, tr_mag, tr_sisdr, tr_base, tr_sil = train_one_epoch(model, optimizer, train_loader, epoch)
        va_loss, va_mag, va_sisdr, va_base, va_sil = validate(model, val_loader, epoch)

        tr_sisdr_db = -tr_sisdr
        va_sisdr_db = -va_sisdr
        tr_base_db = -tr_base
        va_base_db = -va_base

        tr_imp_db = tr_sisdr_db - tr_base_db
        va_imp_db = va_sisdr_db - va_base_db

        print(
            f'Epoch {epoch:03d} | '
            f'train: loss={tr_loss:.4f}, sil={tr_sil:.4f}, base={tr_base_db:.2f}dB, enh={tr_sisdr_db:.2f}dB, imp={tr_imp_db:.2f}dB | '
            f'val: loss={va_loss:.4f}, sil={va_sil:.4f}, base={va_base_db:.2f}dB, enh={va_sisdr_db:.2f}dB, imp={va_imp_db:.2f}dB'
        )

        if SAVE_EVERY_EPOCHS and (epoch % SAVE_EVERY_EPOCHS == 0):
            path = save_checkpoint(epoch, model, optimizer)
            print(f'Saved checkpoint: {path}')

        # Early stopping on validation improvement (higher is better)
        if EARLY_STOPPING:
            if va_imp_db > best_val_imp_db + EARLY_STOP_MIN_DELTA_DB:
                best_val_imp_db = va_imp_db
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(
                    f'Early stopping: no val imp_db improvement for {EARLY_STOP_PATIENCE} epochs. '
                    f'Best val imp_db={best_val_imp_db:.2f}dB'
                )
                break

    print('Done training.')
    return test_loader  # Return test_loader for evaluation

## Inference (Enhance MVDR wav)

This section loads a checkpoint you specify and enhances a single-channel MVDR output wav.
- Input: MVDR wav path (mono)
- Output: enhanced wav saved into `OUTPUT_DIR`

In [13]:
def _save_wav(path: str, wav: torch.Tensor, sr: int):
    """Save mono wav tensor [T] to a .wav file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)

    wav = wav.detach().cpu().to(torch.float32)
    if wav.dim() != 1:
        wav = wav.view(-1)

    if _HAS_SOUNDFILE:
        sf.write(path, wav.numpy(), sr)
        return

    if _HAS_TORCHAUDIO:
        torchaudio.save(path, wav.unsqueeze(0), sr)
        return

    raise ModuleNotFoundError("Missing audio backend for saving. Install `soundfile` or `torchaudio`.")


def load_crn_from_checkpoint(checkpoint_path: str, device: str = DEVICE) -> nn.Module:
    """Load CRN_Light from a checkpoint produced by this notebook."""
    model = CRN_Light().to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)

    # Supports both full checkpoint dict and raw state_dict.
    if isinstance(ckpt, dict) and "model_state" in ckpt:
        state = ckpt["model_state"]
    else:
        state = ckpt

    model.load_state_dict(state)
    model.eval()
    return model


@torch.no_grad()
def enhance_mvdr_waveform(mvdr_wav: torch.Tensor, model: nn.Module, sample_rate: int = SAMPLE_RATE, device: str = DEVICE) -> torch.Tensor:
    """Enhance a mono MVDR waveform.

    Args:
        mvdr_wav: Tensor [T] (mono)
        model: CRN_Light (expects complex-mask input)

    Returns:
        enhanced_wav: Tensor [T]
    """
    if mvdr_wav.dim() != 1:
        mvdr_wav = mvdr_wav.view(-1)

    mvdr_wav = mvdr_wav.to(device)

    # Match training-time scaling (dataset normalizes by MVDR std)
    scale = mvdr_wav.std() + 1e-8
    mvdr_norm = mvdr_wav / scale

    # STFT expects [B, T]
    _, mvdr_complex = stft_mag(mvdr_norm.unsqueeze(0), device=device)

    # Build model input: [real, imag] => [B,2,F,T]
    x = torch.stack([mvdr_complex.real, mvdr_complex.imag], dim=1)

    m = model(x)
    m_r = m[:, 0, :, :]
    m_i = m[:, 1, :, :]

    x_r = mvdr_complex.real
    x_i = mvdr_complex.imag

    y_r = m_r * x_r - m_i * x_i
    y_i = m_r * x_i + m_i * x_r
    est_complex = torch.complex(y_r, y_i)

    est_wav = istft(est_complex, device=device, length=mvdr_norm.shape[-1]).squeeze(0)

    # De-normalize back to original MVDR scale
    est_wav = est_wav * scale
    return est_wav


def enhance_mvdr_file(
    mvdr_wav_path: str,
    checkpoint_path: str,
    output_dir: str,
    sample_rate: int = SAMPLE_RATE,
    output_suffix: str = "_enhanced",
) -> str:
    """Enhance one MVDR wav file and save into output_dir."""
    model = load_crn_from_checkpoint(checkpoint_path, device=DEVICE)

    mvdr = _load_wav(mvdr_wav_path, target_sr=sample_rate)
    enhanced = enhance_mvdr_waveform(mvdr, model=model, sample_rate=sample_rate, device=DEVICE)

    base = os.path.splitext(os.path.basename(mvdr_wav_path))[0]
    out_path = os.path.join(output_dir, f"{base}{output_suffix}.wav")
    _save_wav(out_path, enhanced, sample_rate)
    return out_path


def enhance_mvdr_directory(
    mvdr_dir: str,
    checkpoint_path: str,
    output_dir: str,
    pattern: str = "*.wav",
    sample_rate: int = SAMPLE_RATE,
    output_suffix: str = "_enhanced",
):
    """Enhance all wavs in a directory and save into output_dir."""
    model = load_crn_from_checkpoint(checkpoint_path, device=DEVICE)

    paths = sorted(glob.glob(os.path.join(mvdr_dir, pattern)))
    if not paths:
        raise FileNotFoundError(f"No files matched {pattern} in {mvdr_dir}")

    os.makedirs(output_dir, exist_ok=True)

    pbar = _make_pbar(paths, desc="Enhance")
    for p in pbar:
        mvdr = _load_wav(p, target_sr=sample_rate)
        enhanced = enhance_mvdr_waveform(mvdr, model=model, sample_rate=sample_rate, device=DEVICE)

        base = os.path.splitext(os.path.basename(p))[0]
        out_path = os.path.join(output_dir, f"{base}{output_suffix}.wav")
        _save_wav(out_path, enhanced, sample_rate)

    return output_dir


In [14]:
# Example usage (single file)
# checkpoint_path = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/custom_model_1/checkpoints/v1/crn_epoch_048.pt'
# mvdr_path = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/beamforming/Test_output/COMPARE_V4_mvdr_emon_output.wav'
# OUTPUT_DIR = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/custom_model_1/model_enhanced_output/v1'
# out_path = enhance_mvdr_file(mvdr_path, checkpoint_path, OUTPUT_DIR)
# print('Saved:', out_path)

# Example usage (directory)
checkpoint_path = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/custom_model_1/checkpoints/v2/crn_epoch_060.pt'
OUTPUT_DIR = './model_v2_batch_output'
MVDR_DIR = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/test/noisy'
enhance_mvdr_directory(MVDR_DIR, checkpoint_path, OUTPUT_DIR)
print('Saved enhanced wavs to:', OUTPUT_DIR)


FileNotFoundError: No files matched *.wav in /Users/emonchowdhury/Desktop/Phase 2/av_zoom/DATASET/prepared_dataset/test/noisy

## Test Set Evaluation

Run this section to evaluate a trained model on the saved test set.
- Loads test set indices from `TEST_SET_SAVE_PATH`
- Computes SI-SDR improvement metrics on the held-out test samples

In [15]:
@torch.no_grad()
def evaluate_test_set(
    checkpoint_path: str,
    test_set_manifest_path: str = TEST_SET_MANIFEST_PATH,
    device: str = DEVICE,
):
    """Evaluate model on the saved test set files.
    
    Args:
        checkpoint_path: Path to model checkpoint
        test_set_manifest_path: Path to saved test set manifest JSON
        device: Device to run on
        
    Returns:
        dict with evaluation metrics
    """
    # Load test set manifest
    manifest = load_test_set_info(test_set_manifest_path)
    test_files = manifest['test_files']
    
    # Use saved directories (actual copied files)
    mvdr_dir = manifest.get('saved_mvdr_dir', TEST_SET_MVDR_DIR)
    clean_dir = manifest.get('saved_clean_dir', TEST_SET_CLEAN_DIR)
    
    print(f"Loaded test set: {len(test_files)} samples")
    print(f"  MVDR dir:  {mvdr_dir}")
    print(f"  Clean dir: {clean_dir}")
    
    # Load model
    model = load_crn_from_checkpoint(checkpoint_path, device=device)
    model.eval()
    
    # Metrics accumulators
    results = []
    total_base_sisdr = 0.0
    total_enh_sisdr = 0.0
    
    pbar = _make_pbar(test_files, desc='Test Eval')
    for item in pbar:
        mvdr_path = os.path.join(mvdr_dir, item['mvdr_file'])
        clean_path = os.path.join(clean_dir, item['clean_file'])
        
        if not os.path.isfile(mvdr_path) or not os.path.isfile(clean_path):
            print(f"  Warning: missing file for {item['mvdr_file']}, skipping")
            continue
        
        # Load audio
        mvdr_wav = _load_wav(mvdr_path, SAMPLE_RATE).to(device)
        clean_wav = _load_wav(clean_path, SAMPLE_RATE).to(device)
        
        # Match lengths
        min_len = min(mvdr_wav.shape[-1], clean_wav.shape[-1])
        mvdr_wav = mvdr_wav[:min_len]
        clean_wav = clean_wav[:min_len]
        
        # Normalize (same as training)
        scale = mvdr_wav.std() + 1e-8
        mvdr_norm = mvdr_wav / scale
        clean_norm = clean_wav / scale
        
        # Get baseline SI-SDR
        base_sisdr_loss = si_sdr_loss(mvdr_norm.unsqueeze(0), clean_norm.unsqueeze(0))
        base_sisdr_db = -float(base_sisdr_loss.item())
        
        # Enhance (uses original mvdr_wav, handles normalization internally)
        enhanced_wav = enhance_mvdr_waveform(mvdr_wav, model, device=device)
        enhanced_norm = enhanced_wav / scale
        
        # Get enhanced SI-SDR
        enh_sisdr_loss = si_sdr_loss(enhanced_norm.unsqueeze(0), clean_norm.unsqueeze(0))
        enh_sisdr_db = -float(enh_sisdr_loss.item())
        
        imp_db = enh_sisdr_db - base_sisdr_db
        
        results.append({
            'mvdr_file': item['mvdr_file'],
            'clean_file': item['clean_file'],
            'base_sisdr_db': base_sisdr_db,
            'enh_sisdr_db': enh_sisdr_db,
            'improvement_db': imp_db,
        })
        
        total_base_sisdr += base_sisdr_db
        total_enh_sisdr += enh_sisdr_db
        
        pbar.set_postfix(
            base=f'{total_base_sisdr/len(results):.2f}dB',
            enh=f'{total_enh_sisdr/len(results):.2f}dB',
            imp=f'{(total_enh_sisdr - total_base_sisdr)/len(results):.2f}dB',
        )
    
    n = len(results)
    avg_base = total_base_sisdr / n
    avg_enh = total_enh_sisdr / n
    avg_imp = avg_enh - avg_base
    
    summary = {
        'n_samples': n,
        'avg_baseline_sisdr_db': avg_base,
        'avg_enhanced_sisdr_db': avg_enh,
        'avg_improvement_db': avg_imp,
        'per_sample_results': results,
    }
    
    print(f"\n{'='*50}")
    print(f"TEST SET RESULTS ({n} samples)")
    print(f"{'='*50}")
    print(f"  Baseline SI-SDR:  {avg_base:.2f} dB")
    print(f"  Enhanced SI-SDR:  {avg_enh:.2f} dB")
    print(f"  Improvement:      {avg_imp:.2f} dB")
    print(f"{'='*50}")
    
    return summary


def save_test_results(results: dict, save_path: str):
    """Save test evaluation results to JSON."""
    os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else '.', exist_ok=True)
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Saved test results to: {save_path}")

In [16]:
# Example: Evaluate on test set
# Uncomment and run after training is complete

# checkpoint_path = 'checkpoints/crn_latest.pt'  # or specific epoch
# test_results = evaluate_test_set(checkpoint_path)

# Optionally save results
# save_test_results(test_results, 'checkpoints/test_results.json')

In [ ]:
# Run training
# main()

Pairing sanity check (first few samples):
  1001_part1_A_female_only.wav -> 1001_part1.flac
  1001_part1_B_female_music.wav -> 1001_part1.flac
  1001_part1_C_female_noise.wav -> 1001_part1.flac
  1001_part1_D_music_noise.wav -> 1001_part1.flac
  1001_part1_E_female_music_noise.wav -> 1001_part1.flac
Device: cuda
Train/Val sizes: 37702/6653
Val full utterance: True
Total parameters: 0.13M
Start epoch: 1 / Target epochs: 60


Train 001:   0%|          | 0/4713 [00:00<?, ?it/s]

KeyboardInterrupt: 

Edit [](file:///Users/emonchowdhury/Desktop/kaggle-crn-light-merged-train-v2.ipynb)

Now let me update the test evaluation cell to use the saved files: 

Edit [](file:///Users/emonchowdhury/Desktop/kaggle-crn-light-merged-train-v2.ipynb)

Done! Now the test set saves the **actual audio files** instead of just indices. Here's what changed:

### Updated Test Set Saving

**New directory structure:**


In [ ]:
checkpoints/
└── test_set/
    ├── mvdr/          # Copied MVDR audio files
    │   ├── 1_part13_A_female_only.wav
    │   └── ...
    ├── clean/         # Copied clean reference audio files
    │   ├── 1_part13.wav
    │   └── ...
    └── manifest.json  # Metadata + file list



**Key changes in cell 12:**
- Added `shutil` import for file copying
- New config paths: `TEST_SET_DIR`, `TEST_SET_MVDR_DIR`, `TEST_SET_CLEAN_DIR`, `TEST_SET_MANIFEST_PATH`
- `save_test_set_files()` now copies actual audio files using `shutil.copy2()` (preserves metadata)
- Manifest JSON includes paths to saved directories

**Updated test evaluation (cell 16):**
- `evaluate_test_set()` now loads audio directly from the saved test set directories
- Works independently of original dataset paths (portable for later testing)

Made changes.

## Combined Pipeline Report: MVDR + CRN_Light + Video

### Workflow
1. **MATLAB**: Set `K_selected` in `benchmark_mvdr_latency.m` → run → copy the two output lines
2. **Below**: Paste `mvdr_K` and `mvdr_ms_per_frame` → run → get final combined report

### Pipeline (MVDR N=512 aligned — no redundant STFT/iSTFT)
```
Mic Array → STFT(N=512) → MVDR beamform → CRN_Light model → iSTFT → Speaker
                               ↑ shared STFT — no redundant transforms ↑
```

In [17]:
import time

def combined_pipeline_report_crn(
    checkpoint_path: str,
    # --- PASTE FROM MATLAB ---
    mvdr_K: int = 4,
    mvdr_ms_per_frame: float = 0.8511,
    # --- Estimation factors ---
    mvdr_smartphone_factor: float = 3.0,
    model_smartphone_factor: float = 3.0,
    # --- Video pipeline ---
    video_fps: int = 30,
    video_frame_processing_ms: float = 10.0,
    # --- Benchmark config ---
    n_warmup: int = 5,
    n_runs: int = 50,
    # --- Threading ---
    parallel_av: bool = True,
):
    """Final combined pipeline report for MVDR + CRN_Light + Video.
    
    Takes MVDR per-frame timing (from MATLAB benchmark) + measures CRN_Light
    latency here, then produces the full real-time feasibility verdict.
    """
    device = 'cpu'
    model = load_crn_from_checkpoint(checkpoint_path, device=device)
    model.eval()

    n_fft = 512
    hop_length = 128
    sample_rate = SAMPLE_RATE  # 16000

    hop_budget_ms = (hop_length / sample_rate) * 1000  # 8.0 ms
    video_budget_ms = 1000.0 / video_fps               # 33.3 ms @ 30fps
    hops_per_video_frame = video_budget_ms / hop_budget_ms

    # Count parameters
    n_params = sum(p.numel() for p in model.parameters())
    n_params_str = f'{n_params / 1e6:.3f}M'

    # =========================================================================
    # 1) Measure CRN_Light (model + iSTFT, direct STFT feed from MVDR)
    # =========================================================================
    test_dur = 1.0
    n_samples = int(sample_rate * test_dur)
    n_frames = (n_samples + hop_length - 1) // hop_length

    # Simulate MVDR STFT output: complex [1, 257, T]
    spec_mvdr = torch.randn(1, n_fft // 2 + 1, n_frames, dtype=torch.cfloat, device=device)

    # Build model input: [B, 2, F, T] -> [real, imag]
    x_bench = torch.stack([spec_mvdr.real, spec_mvdr.imag], dim=1)

    for _ in range(n_warmup):
        with torch.no_grad():
            mask = model(x_bench)
            m_r = mask[:, 0, :, :]
            m_i = mask[:, 1, :, :]
            y_r = m_r * spec_mvdr.real - m_i * spec_mvdr.imag
            y_i = m_r * spec_mvdr.imag + m_i * spec_mvdr.real
            enh = torch.complex(y_r, y_i)
            _ = istft(enh, device, length=n_samples)

    model_times = []
    for _ in range(n_runs):
        with torch.no_grad():
            t0 = time.perf_counter()
            mask = model(x_bench)
            m_r = mask[:, 0, :, :]
            m_i = mask[:, 1, :, :]
            y_r = m_r * spec_mvdr.real - m_i * spec_mvdr.imag
            y_i = m_r * spec_mvdr.imag + m_i * spec_mvdr.real
            enh = torch.complex(y_r, y_i)
            _ = istft(enh, device, length=n_samples)
            t1 = time.perf_counter()
        model_times.append((t1 - t0) * 1000)

    model_total_ms = np.mean(model_times)
    model_per_frame_desk = model_total_ms / n_frames
    model_per_frame_phone = model_per_frame_desk * model_smartphone_factor

    # =========================================================================
    # 2) MVDR phone estimate
    # =========================================================================
    mvdr_phone = mvdr_ms_per_frame * mvdr_smartphone_factor

    # =========================================================================
    # 3) Combined
    # =========================================================================
    combined_desk = mvdr_ms_per_frame + model_per_frame_desk
    combined_phone = mvdr_phone + model_per_frame_phone
    audio_ok = combined_phone < hop_budget_ms

    if audio_ok:
        headroom = ((hop_budget_ms - combined_phone) / hop_budget_ms) * 100
    else:
        headroom = -((combined_phone - hop_budget_ms) / hop_budget_ms) * 100

    # AV feasibility
    audio_per_vframe = combined_phone * hops_per_video_frame
    if parallel_av:
        av_bottleneck = max(audio_per_vframe, video_frame_processing_ms)
        av_ok = av_bottleneck < video_budget_ms
    else:
        av_total = audio_per_vframe + video_frame_processing_ms
        av_ok = av_total < video_budget_ms

    # =========================================================================
    # REPORT
    # =========================================================================
    print(f'{"=" * 72}')
    print(f'  COMBINED PIPELINE REPORT: MVDR + CRN_Light + Video')
    print(f'  Audiovisual Zooming — Smartphone Real-Time Feasibility')
    print(f'{"=" * 72}')

    print(f'\n  Configuration:')
    print(f'    MVDR  : N=512, hop=128, nfft=512, 2 mics, K={mvdr_K}')
    print(f'    Model : CRN_Light, {n_params_str} params, CPU')
    print(f'    Video : {video_fps} fps, ~{video_frame_processing_ms:.0f} ms/frame')
    print(f'    STFT  : Aligned N=512 (direct feed, no redundant STFT/iSTFT)')
    print(f'    Thread: {"Parallel (audio+video on separate cores)" if parallel_av else "Sequential"}')

    print(f'\n  MVDR (from MATLAB benchmark, K={mvdr_K}):')
    print(f'    Desktop : {mvdr_ms_per_frame:.4f} ms/frame')
    print(f'    Phone   : {mvdr_phone:.4f} ms/frame  (×{mvdr_smartphone_factor})')

    print(f'\n  CRN_Light (measured, {n_frames} frames):')
    print(f'    Desktop : {model_per_frame_desk:.4f} ms/frame')
    print(f'    Phone   : {model_per_frame_phone:.4f} ms/frame  (×{model_smartphone_factor})')

    print(f'\n{"─" * 72}')
    print(f'  AUDIO BUDGET (per hop = {hop_budget_ms:.1f} ms)')
    print(f'{"─" * 72}')
    print(f'  {"Component":<32s}  {"Desktop":>10s}  {"Phone":>10s}  {"Budget":>8s}')
    print(f'  {"─"*32}  {"─"*10}  {"─"*10}  {"─"*8}')
    print(f'  {"MVDR (K=" + str(mvdr_K) + ")":<32s}  {mvdr_ms_per_frame:>9.4f}  {mvdr_phone:>9.4f}  {hop_budget_ms:>7.1f}')
    print(f'  {"CRN_Light (model+iSTFT)":<32s}  {model_per_frame_desk:>9.4f}  {model_per_frame_phone:>9.4f}  {hop_budget_ms:>7.1f}')
    print(f'  {"─"*32}  {"─"*10}  {"─"*10}  {"─"*8}')
    print(f'  {"COMBINED AUDIO":<32s}  {combined_desk:>9.4f}  {combined_phone:>9.4f}  {hop_budget_ms:>7.1f}')
    print(f'  Headroom: {headroom:+.0f}%  {"✅ REAL-TIME" if audio_ok else "❌ OVER BUDGET"}')

    # AV section
    print(f'\n{"─" * 72}')
    print(f'  AV BUDGET (per video frame = {video_budget_ms:.1f} ms @ {video_fps}fps)')
    print(f'{"─" * 72}')
    print(f'  Audio ({hops_per_video_frame:.1f} hops/vframe) : {audio_per_vframe:.2f} ms')
    print(f'  Video processing            : {video_frame_processing_ms:.2f} ms')
    if parallel_av:
        print(f'  Parallel max (bottleneck)   : {av_bottleneck:.2f} ms / {video_budget_ms:.1f} ms  '
              f'{"✅" if av_ok else "❌"}')
    else:
        print(f'  Sequential total            : {av_total:.2f} ms / {video_budget_ms:.1f} ms  '
              f'{"✅" if av_ok else "❌"}')

    # Latency split bar
    mvdr_pct = mvdr_phone / combined_phone * 100
    model_pct = 100 - mvdr_pct
    bar_w = 50
    mvdr_bar = max(1, int(mvdr_pct / 100 * bar_w))
    model_bar = bar_w - mvdr_bar
    used_pct = combined_phone / hop_budget_ms * 100
    used_bar = min(bar_w, max(1, int(used_pct / 100 * bar_w)))
    free_bar = bar_w - used_bar

    print(f'\n  ⚡ Latency Split (phone):')
    print(f'    MVDR  [{"█" * mvdr_bar}{"░" * model_bar}] {mvdr_pct:.0f}% ({mvdr_phone:.4f} ms)')
    print(f'    Model [{"░" * mvdr_bar}{"█" * model_bar}] {model_pct:.0f}% ({model_per_frame_phone:.4f} ms)')
    print(f'    Total [{"█" * used_bar}{"░" * free_bar}] {combined_phone:.4f} / {hop_budget_ms:.1f} ms ({headroom:+.0f}%)')

    # Final verdict
    print(f'\n{"=" * 72}')
    print(f'  VERDICT')
    print(f'{"=" * 72}')

    if audio_ok and av_ok:
        print(f'\n  ✅ REAL-TIME FEASIBLE')
        print(f'     Combined audio: {combined_phone:.4f} ms/frame ({headroom:+.0f}% headroom)')
        print(f'     Audio RTF (phone): {combined_phone / hop_budget_ms:.4f}')
        print(f'     MVDR weights update every {mvdr_K} frames = every {mvdr_K * hop_budget_ms:.0f} ms')
        if mvdr_K <= 1:
            print(f'\n  🎯 K=1: Max adaptivity, tracks fast-moving sources in real time')
        elif mvdr_K <= 4:
            print(f'\n  🎯 K={mvdr_K}: Great balance — fast tracking + plenty of headroom')
        elif mvdr_K <= 8:
            print(f'\n  🎯 K={mvdr_K}: Good for slowly-moving or stationary sources')
        else:
            print(f'\n  🎯 K={mvdr_K}: Minimal cost but slower source tracking')
    elif audio_ok and not av_ok:
        print(f'\n  ⚠️  Audio OK but AV pipeline exceeds budget')
        print(f'     Consider: lighter video model or lower fps')
    else:
        print(f'\n  ❌ NOT FEASIBLE with K={mvdr_K}')
        print(f'     Try: larger K, INT8 quantization, CoreML/NNAPI, larger hop')

    print(f'\n{"=" * 72}')

    return {
        'mvdr_K': mvdr_K,
        'mvdr_desk_ms': mvdr_ms_per_frame,
        'mvdr_phone_ms': mvdr_phone,
        'model_desk_ms': model_per_frame_desk,
        'model_phone_ms': model_per_frame_phone,
        'combined_desk_ms': combined_desk,
        'combined_phone_ms': combined_phone,
        'headroom_pct': headroom,
        'audio_ok': audio_ok,
        'av_ok': av_ok,
    }

In [18]:
# ═══════════════════════════════════════════════════════════
# PASTE FROM MATLAB benchmark_mvdr_latency.m output below:
# ═══════════════════════════════════════════════════════════
mvdr_K = 4
mvdr_ms_per_frame = 0.8511
# ═══════════════════════════════════════════════════════════

pipeline_results_crn = combined_pipeline_report_crn(
    checkpoint_path='/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/custom_model_1/checkpoints/v2/crn_epoch_060.pt',
    mvdr_K=mvdr_K,
    mvdr_ms_per_frame=mvdr_ms_per_frame,
)

  COMBINED PIPELINE REPORT: MVDR + CRN_Light + Video
  Audiovisual Zooming — Smartphone Real-Time Feasibility

  Configuration:
    MVDR  : N=512, hop=128, nfft=512, 2 mics, K=4
    Model : CRN_Light, 0.134M params, CPU
    Video : 30 fps, ~10 ms/frame
    STFT  : Aligned N=512 (direct feed, no redundant STFT/iSTFT)
    Thread: Parallel (audio+video on separate cores)

  MVDR (from MATLAB benchmark, K=4):
    Desktop : 0.8511 ms/frame
    Phone   : 2.5533 ms/frame  (×3.0)

  CRN_Light (measured, 125 frames):
    Desktop : 0.4729 ms/frame
    Phone   : 1.4186 ms/frame  (×3.0)

────────────────────────────────────────────────────────────────────────
  AUDIO BUDGET (per hop = 8.0 ms)
────────────────────────────────────────────────────────────────────────
  Component                            Desktop       Phone    Budget
  ────────────────────────────────  ──────────  ──────────  ────────
  MVDR (K=4)                           0.8511     2.5533      8.0
  CRN_Light (model+iSTFT)        